# LangChain 输出解析器：一张借书单，四种用法

聊天模型返回一条 `AIMessage`。提示词告诉模型按什么格式回答，输出解析器再把回复里的文本变成 Python 可以使用的数据。

下面用同一句借书请求做练习。选择 `langchain_env` 内核，按顺序运行。前三个例子各调用一次 `gpt-6-luna`，第四个复用第三个的回复。

In [1]:
from dotenv import dotenv_values
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import (
    StrOutputParser, CommaSeparatedListOutputParser, JsonOutputParser, PydanticOutputParser,
)

config = dotenv_values(".env")
if not config.get("OPENAI_API_KEY") or not config.get("OPENAI_PROXY"):
    raise ValueError("请在本目录的 .env 中填写 OPENAI_API_KEY 和 OPENAI_PROXY。")

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=config["OPENAI_API_KEY"],
    openai_proxy=config["OPENAI_PROXY"],
    base_url="https://api.openai.com/v1",
    use_responses_api=True,
    max_completion_tokens=512,  # 每次生成最多 512 个 token，包含推理和回复。
)
note = "小林想借《西游记》和《三国演义》，借 7 天。"

def ask(rule):
    # 每次只换输出要求，用户提供的借书信息相同。
    return model.invoke([("system", rule), ("human", note)])

## 1. 字符串 `str`：显示给人看

聊天回复、摘要和通知都可以直接显示为文本。下面先看原始消息的类型，再取出可以打印的确认语。

In [2]:
message = ask("用一句简短的中文确认这张借书单。")
confirmation = StrOutputParser().invoke(message)

print("消息类型：", type(message).__name__)
print("解析结果：", type(confirmation).__name__, confirmation)

消息类型： AIMessage
解析结果： TextAccessor 确认借阅《西游记》和《三国演义》，借期 7 天。


## 2. 列表 `list`：逐个处理

如果要逐本查询库存，先提取书名列表，再用 `for` 遍历。这个解析器按英文逗号分隔，所以提示词也要约定分隔符。

In [3]:
message = ask("只输出书名，保持原顺序，用英文逗号分隔，省略书名号。")
titles = CommaSeparatedListOutputParser().invoke(message)

print(titles)
for title in titles:
    print("待查库存：", title)

['西游记', '三国演义']
待查库存： 西游记
待查库存： 三国演义


## 3. 字典 `dict`：按字段取值

要填借书表单时，需要把借阅人、书名和天数分开。让模型输出一个 JSON 对象，解析后就可以按键读取；JSON 数组和数字也会变成 Python 的列表和数字。

In [4]:
json_message = ask("只输出一个 JSON 对象。borrower 是借阅人，books 是书名数组，days 是借阅天数（整数）。")
record = JsonOutputParser().invoke(json_message)

print("原文：", json_message.text)
print("借阅人：", record["borrower"])
print("借阅天数：", record["days"], type(record["days"]).__name__)

原文： {"borrower":"小林","books":["西游记","三国演义"],"days":7}
借阅人： 小林
借阅天数： 7 int


## 4. Pydantic 对象：写入系统前校验规则

一份 JSON 能读出来，借阅天数仍可能写成 0。用 Pydantic 声明字段和规则，校验通过后就能用 `loan.days` 读取天数。这里复用上一格的 `json_message`。

In [5]:
from pydantic import BaseModel, Field
from langchain_core.exceptions import OutputParserException

class Loan(BaseModel):
    borrower: str
    books: list[str]
    days: int = Field(gt=0)  # gt 表示“大于”，借阅天数至少为 1。

loan_parser = PydanticOutputParser(pydantic_object=Loan)
loan = loan_parser.invoke(json_message)
print(f"{loan.borrower}借了{len(loan.books)}本书，借期{loan.days}天。")

# 本地反例：这段文本符合 JSON 语法，但违反了上面的天数规则。
try:
    loan_parser.parse('{"borrower":"小林","books":["西游记"],"days":0}')
except OutputParserException:
    print("校验未通过：借阅天数必须大于 0。")

小林借了2本书，借期7天。
校验未通过：借阅天数必须大于 0。


### 如果消息里有工具调用或图片呢？

上面四个例子都在解析**回复里的文本**。同一条消息还可以携带其他内容：

| 想做的事 | 在 LangChain 中怎么取 |
|---|---|
| 读取模型请求调用的工具，例如查库存 | 从 `message.tool_calls` 读取工具名 `name` 和参数 `args`，由程序执行查询。 |
| 读取一条多模态消息里已有的图片 | 遍历 `message.content_blocks`，查看 `type` 为 `image` 的内容块。 |

本课的三次请求只生成文本。进一步阅读：[LangChain 消息与内容块](https://docs.langchain.com/oss/python/langchain/messages)。